# Introducción a las Pruebas de Integración

**Objetivo:** Comprender qué son las pruebas de integración, por qué son necesarias más allá de las pruebas unitarias, y familiarizarse con los distintos enfoques de integración.

---

## 1. ¿Qué es una prueba de integración?

Una **prueba de integración** verifica que dos o más módulos de software, que ya han sido probados de forma aislada (pruebas unitarias), **colaboran correctamente** entre sí. Mientras las pruebas unitarias se centran en la lógica interna de cada componente, las pruebas de integración validan las interfaces, la comunicación y la interacción entre componentes.

> «Las pruebas de integración verifican la comunicación, el flujo de datos y la interacción correcta entre componentes de software previamente verificados de forma unitaria.»  
> — ISTQB Glossary

**Ejemplo:** Un módulo `Service` que utiliza un módulo `Storage` para guardar datos y un módulo `Notifier` para enviar notificaciones. Las pruebas unitarias de cada módulo pueden pasar, pero ¿qué sucede si `Service` pasa un parámetro en formato incorrecto a `Storage`? Eso **solo se detecta con una prueba de integración**.

---

## 2. La pirámide de pruebas

La **pirámide de pruebas** (Mike Cohn, 2009) describe la proporción recomendada de cada tipo de prueba:

```
         /\
        /  \        ← E2E / UI  (pocos, lentos, costosos)
       /────\
      /      \      ← Integration  (moderados)
     /────────\
    /          \    ← Unit  (muchos, rápidos, aislados)
   /────────────\
```

| Nivel | ¿Qué prueba? | Velocidad | Aislamiento |
|-------|-------------|-----------|-------------|
| **Unitario** | Un módulo/función en aislamiento | Muy rápido | Total (mocks/stubs) |
| **Integración** | Colaboración entre 2+ módulos | Medio | Parcial |
| **E2E** | Flujo completo del sistema | Lento | Ninguno |

---

## 3. Limitaciones de las pruebas unitarias

Las pruebas unitarias aíslan cada módulo usando mocks y stubs. Esto permite probar la lógica interna, pero **no garantiza** que los módulos reales funcionen juntos. Problemas típicos que las pruebas unitarias no detectan:

| Tipo de fallo | Descripción | Ejemplo en nuestro sistema |
|---------------|-------------|----------------------------|
| **Incompatibilidad de interfaz** | Tipos o formatos de datos incorrectos | `storage.save()` recibe formato incorrecto desde `service` |
| **Excepción no controlada** | Un módulo lanza excepción que otro no captura | `Notifier.send()` lanza `ConnectionError`; `service` no lo maneja |
| **Estado inconsistente** | Operación parcialmente completada | Tarea guardada pero notificación fallida, sin rollback |
| **Orden incorrecto de llamadas** | Dependencias invocadas en secuencia equivocada | `save()` llamado antes de `load()` al actualizar |
| **Efectos secundarios no previstos** | Una operación altera el estado de otro módulo | `load()` devuelve referencia mutable que `service` modifica |

---

## 4. Arquitectura del sistema de ejemplo

Trabajaremos sobre un sistema gestor de tareas con tres módulos:

```
┌──────────────────────────────────────────┐
│              Cliente / Tests             │
└────────────────────┬─────────────────────┘
                     │ llama a
┌────────────────────▼─────────────────────┐
│           TaskService (service.py)       │  ← Lógica de negocio
│  + add_task(title)                       │
│  + complete_task(title)                  │
│  + list_tasks()                          │
└──────────┬────────────────────┬──────────┘
           │ usa                │ usa
┌──────────▼──────────┐  ┌──────▼──────────┐
│  TaskStorage        │  │    Notifier      │
│  (storage.py)       │  │  (notifier.py)   │
│  + save(tasks)      │  │  + send(message) │
│  + load()           │  │                  │
└─────────────────────┘  └──────────────────┘
         ↕ JSON
    [archivo .json]
```

Las pruebas de integración validan las **flechas** entre cajas, no solo el interior de cada caja.

- **Storage:** Lee y escribe tareas en un archivo JSON.
- **Notifier:** Simula envío de notificaciones (tiene un 10% de fallo aleatorio — ¡un bug de integración intencional!).
- **Service:** Lógica de negocio; recibe peticiones, persiste mediante `Storage` y notifica mediante `Notifier`.

---

## 5. Enfoques de integración — resumen

Existen cuatro estrategias principales para planificar y ejecutar pruebas de integración:

| Enfoque | Dirección | Sustitutos necesarios | Detecta fallos bajos temprano | Complejidad |
|---------|-----------|----------------------|-------------------------------|-------------|
| **Big-Bang** | N/A | Ninguno | No | Baja (pero diagnóstico difícil) |
| **Top-Down** | Arriba → Abajo | Stubs | No | Media |
| **Bottom-Up** | Abajo → Arriba | Drivers | Sí | Media |
| **Sandwich** | Ambas | Stubs + Drivers | Sí | Alta |

En el notebook `03_enfoques.ipynb` exploraremos cada uno a fondo con implementaciones concretas.

---

## 6. Primer ejercicio: analizar las pruebas iniciales

Antes de continuar, ejecuta el proyecto base y observa los resultados.

In [1]:
# Ejecuta las pruebas iniciales y observa que todas pasan
# (Desde la raíz del proyecto: pytest tests/ -v)
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "../tests/", "-v", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

============================= test session starts =============================
platform win32 -- Python 3.11.9, pytest-9.0.3, pluggy-1.6.0 -- c:\Users\User\Documents\Software_Quality_UAN\venv\Scripts\python.exe
cachedir: .pytest_cache
hypothesis profile 'default'
rootdir: c:\Users\User\Documents\Software_Quality_UAN\talleres\integration_testing
plugins: anyio-4.13.0, hypothesis-6.152.4, mock-3.15.1
collecting ... collected 3 items

..\tests\test_service_integration.py::TestServiceIntegration::test_add_task_happy_path PASSED [ 33%]
..\tests\test_service_integration.py::TestServiceIntegration::test_complete_task PASSED [ 66%]
..\tests\test_storage_driver.py::test_storage_save_and_load PASSED       [100%]

============================== 3 passed in 1.16s ==============================




## 7. ¿Por qué pasan estas pruebas pero el sistema tiene errores?

Ejecuta la celda anterior e inspecciona el código de las pruebas en `tests/test_service_integration.py`. Observarás que:

1. Las pruebas usan el `Notifier` **real** (con su fallo aleatorio del 10%).
2. Solo verifican el valor de retorno (`result is True`), **no el estado** de `storage` después de la operación.
3. No simulan ningún escenario de fallo.

Responde en tu informe:
- ¿Las pruebas actuales verifican realmente que `Service` y `Storage` colaboran correctamente?
- ¿Qué interacciones entre módulos no están siendo validadas?
- ¿Qué pasaría si `add_task()` guardara la tarea pero nunca llamara a `notifier.send()`?

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import inspect
from src.service import TaskService
from src.storage import TaskStorage
from src.notifier import Notifier

# Inspeccionamos las interfaces públicas de cada módulo
print("=== Interfaces del sistema ===\n")
for cls in [TaskService, TaskStorage, Notifier]:
    print(f"--- {cls.__name__} ---")
    for name, method in inspect.getmembers(cls, predicate=inspect.isfunction):
        if not name.startswith('_'):
            sig = inspect.signature(method)
            doc = (inspect.getdoc(method) or "").split('\n')[0]
            print(f"  {name}{sig}")
            if doc:
                print(f"    → {doc}")
    print()

=== Interfaces del sistema ===

--- TaskService ---
  add_task(self, title)
    → Agrega una nueva tarea.
  complete_task(self, title)
    → Marca una tarea como completada.
  list_tasks(self)

--- TaskStorage ---
  load(self)
    → Carga la lista de tareas. Retorna lista vacía si el archivo no existe.
  save(self, tasks)
    → Guarda la lista de tareas en el archivo.

--- Notifier ---
  send(self, message)
    → Envía una notificación.



## 8. Demostración: lo que una prueba unitaria NO detecta

El siguiente código muestra cómo una prueba unitaria típica con mocks pasa correctamente, pero **no detecta** cambios en el contrato entre módulos.

In [3]:
from unittest.mock import MagicMock

# Prueba unitaria típica — todo mockeado
mock_storage = MagicMock()
mock_storage.load.return_value = []
mock_storage.save.return_value = None

mock_notifier = MagicMock()
mock_notifier.send.return_value = None

service = TaskService(mock_storage, mock_notifier)
result = service.add_task("Estudiar integración")

print(f"Resultado: {result}")
print(f"save() llamado con: {mock_storage.save.call_args}")
print()
print("OBSERVACIÓN: Esta prueba pasa aunque cambiemos el contrato de save().")
print("Los mocks no reflejan la implementación real de TaskStorage.")
print("Si storage.save() cambia su firma, esta prueba unitaria SIGUE PASANDO.")

Resultado: True
save() llamado con: call([{'title': 'Estudiar integración', 'done': False}])

OBSERVACIÓN: Esta prueba pasa aunque cambiemos el contrato de save().
Los mocks no reflejan la implementación real de TaskStorage.
Si storage.save() cambia su firma, esta prueba unitaria SIGUE PASANDO.
